# Math Concepts for Neural Networks 

This notebook builds *the minimum math intuition* you need for backprop and training.

**Goal:** you can compute (and reason about) gradients by hand for a single neuron, and you can interpret them.


## Setup

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)


## 1) Scalars, vectors, matrices: shapes that matter

Neural nets are just repeated applications of:
- **weighted sums** (dot products / matrix multiplies)
- **nonlinearities** (e.g., sigmoid, ReLU)
- **loss** (a scalar that we minimize)

The main skill: keep track of **shapes**.


In [ ]:
# Scalar
a = 3.0

# Vector (shape: (d,))
x = np.array([1.0, 2.0, -1.0])

# Matrix (shape: (m, d))
W = np.array([
    [0.2, -0.1, 0.4],
    [0.0,  0.3, -0.2]
])

b = np.array([0.1, -0.3])  # (m,)

print("a:", a, "shape:", np.shape(a))
print("x:", x, "shape:", x.shape)
print("W:\n", W, "shape:", W.shape)
print("b:", b, "shape:", b.shape)

# Linear layer: z = W x + b  -> shape (m,)
z = W @ x + b
print("\nz = W @ x + b:", z, "shape:", z.shape)


### Exercise
Change `x` to a different length and watch how it breaks. Fix it by updating `W`.


## 2) Dot product = weighted sum (and similarity)

For vectors `w` and `x`, the dot product `w·x = Σ w_i x_i`.

In a neuron, this is the **pre-activation** before adding bias and applying activation.


In [ ]:
w = np.array([0.5, -1.0, 2.0])
x = np.array([1.5, 0.2, -0.3])

dot = w @ x
manual = (w[0]*x[0]) + (w[1]*x[1]) + (w[2]*x[2])

print("w:", w)
print("x:", x)
print("w @ x:", dot)
print("manual:", manual)


## 3) Key functions: sigmoid and ReLU

You should know their shapes and derivatives.

- Sigmoid: `σ(z) = 1/(1+e^{-z})`, derivative: `σ(z)(1-σ(z))`
- ReLU: `max(0,z)`, derivative: `1` if `z>0` else `0`


In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def d_sigmoid_from_sig(sig):
    # If sigmoid(z) = sig, derivative is sig*(1-sig)
    return sig * (1.0 - sig)

def relu(z):
    return np.maximum(0.0, z)

def d_relu(z):
    return (z > 0).astype(float)

zs = np.linspace(-8, 8, 401)
sig = sigmoid(zs)
dsig = d_sigmoid_from_sig(sig)
r = relu(zs)
dr = d_relu(zs)

plt.figure()
plt.plot(zs, sig, label="sigmoid")
plt.plot(zs, dsig, label="sigmoid'")
plt.legend()
plt.title("Sigmoid and its derivative")
plt.xlabel("z")
plt.ylabel("value")
plt.show()

plt.figure()
plt.plot(zs, r, label="relu")
plt.plot(zs, dr, label="relu'")
plt.legend()
plt.title("ReLU and its derivative")
plt.xlabel("z")
plt.ylabel("value")
plt.show()


## 4) Loss functions (scalar objectives)

A **loss** is a single number that tells you how wrong the model is.

Two common ones:
- **MSE** for regression: `L = 0.5 (ŷ - y)^2`
- **Binary cross-entropy** for classification: `L = -[y log(ŷ) + (1-y) log(1-ŷ)]`

For Phase 1, we’ll use **MSE** because the derivatives are super clean.


In [ ]:
def mse_loss(y_hat, y):
    return 0.5 * (y_hat - y)**2

y_hat = 0.7
y = 1.0
print("MSE loss:", mse_loss(y_hat, y))


## 5) Chain rule: the entire secret of backprop

If `L` depends on `a`, and `a` depends on `z`, then:

`dL/dz = (dL/da) * (da/dz)`

Backprop is just applying this repeatedly through the computation graph.


### A simple computation graph

Let:
- `z = w·x + b`
- `a = σ(z)`
- `L = 0.5 (a - y)^2`

We will compute gradients **by hand**:
- `dL/dw`
- `dL/db`

This is the core you must own.


In [ ]:
def forward(w, b, x):
    z = w @ x + b
    a = sigmoid(z)
    return z, a

def loss_mse(a, y):
    return 0.5 * (a - y)**2

# One data point
x = np.array([1.5, 0.2, -0.3])
y = 1.0

# Parameters
w = np.array([0.5, -1.0, 2.0])
b = -0.1

z, a = forward(w, b, x)
L = loss_mse(a, y)

print("z:", z)
print("a = sigmoid(z):", a)
print("L:", L)


## 6) Manual derivatives for the single neuron

We compute step-by-step:

1) `dL/da = (a - y)`

2) `da/dz = a(1-a)` because `a = sigmoid(z)`

3) `dz/dw = x` and `dz/db = 1` because `z = w·x + b`

Combine with chain rule:

`dL/dz = (a - y) * a(1-a)`

`dL/dw = dL/dz * x`

`dL/db = dL/dz`


In [ ]:
# Manual gradients
dL_da = (a - y)                 # derivative of 0.5(a-y)^2 wrt a
da_dz = a * (1.0 - a)           # sigmoid derivative
dL_dz = dL_da * da_dz

dL_dw = dL_dz * x               # elementwise multiply: each w_i gets x_i scaled by dL/dz
dL_db = dL_dz

print("dL/da:", dL_da)
print("da/dz:", da_dz)
print("dL/dz:", dL_dz)
print("dL/dw:", dL_dw)
print("dL/db:", dL_db)


### Interpret the signs (this is the intuition)

- If `dL/dw_i` is **positive**, decreasing `w_i` decreases loss.
- If `dL/dw_i` is **negative**, increasing `w_i` decreases loss.

In gradient descent: `w := w - lr * dL/dw`.


In [ ]:
lr = 0.5
w_new = w - lr * dL_dw
b_new = b - lr * dL_db

z2, a2 = forward(w_new, b_new, x)
L2 = loss_mse(a2, y)

print("Old L:", float(L))
print("New L:", float(L2))
print("ΔL:", float(L2 - L))


## 7) Numerical gradient check (sanity check)

We approximate gradients by finite differences and compare with the manual ones.

If these match, your chain rule math is correct.


In [ ]:
def loss_given_params(w, b, x, y):
    z = w @ x + b
    a = sigmoid(z)
    return float(0.5 * (a - y)**2)

def numerical_grad_w(w, b, x, y, eps=1e-6):
    g = np.zeros_like(w, dtype=float)
    base = loss_given_params(w, b, x, y)
    for i in range(len(w)):
        w2 = w.copy()
        w2[i] += eps
        g[i] = (loss_given_params(w2, b, x, y) - base) / eps
    return g

def numerical_grad_b(w, b, x, y, eps=1e-6):
    base = loss_given_params(w, b, x, y)
    return (loss_given_params(w, b + eps, x, y) - base) / eps

g_num_w = numerical_grad_w(w, b, x, y)
g_num_b = numerical_grad_b(w, b, x, y)

print("Manual dL/dw:", dL_dw)
print("Numeric dL/dw:", g_num_w)
print("Abs diff:", np.abs(dL_dw - g_num_w))

print("\nManual dL/db:", dL_db)
print("Numeric dL/db:", g_num_b)
print("Abs diff:", abs(dL_db - g_num_b))


## 8) Visualize loss vs a single weight

This helps you see what gradient descent is doing: it’s moving downhill on a loss curve.


In [ ]:
i = 0  # choose which weight to vary
ws = np.linspace(w[i]-3, w[i]+3, 400)

Ls = []
for wi in ws:
    wtmp = w.copy()
    wtmp[i] = wi
    Ls.append(loss_given_params(wtmp, b, x, y))

plt.figure()
plt.plot(ws, Ls)
plt.title(f"Loss vs w[{i}] (holding others fixed)")
plt.xlabel(f"w[{i}]")
plt.ylabel("Loss")
plt.show()

print("Current w[i]:", w[i], "Current loss:", loss_given_params(w, b, x, y))
print("Gradient dL/dw[i]:", dL_dw[i])


## What you should be able to say out loud after this notebook

- A neuron is a dot product + bias + nonlinearity.
- Backprop is chain rule.
- `dL/dw` points in the direction of increasing loss; subtract it to decrease loss.
- Sigmoid can saturate: when `a≈0` or `a≈1`, `a(1-a)` is tiny → vanishing gradients.
